# Introduction
This notebook demonstrates how to set up and run a quantized version of the Llama-3-8B model. We will begin with some basic setup and then proceed to load and use the model.

## 1. Hello World

In [1]:
# First, let's print a simple message to ensure our environment is set up correctly.
print("Hello World")

Hello World


## 2. Initial Setup

In [38]:
import os
#os.chdir('..')
print("Current Working Directory " , os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

import torch
with torch.no_grad():
    torch.cuda.empty_cache()

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

MemTotal: 1007.72 GB
MemFree: 100.64 GB
MemAvailable: 979.06 GB
Free GPU Memory (GB): 39.3896


In [ ]:
# Code formatting and linting

!black notebooks/Llama-3-8B-quant.ipynb
!pylint notebooks/Llama-3-8B-quant.ipynb

In [5]:
# HuggingFace authentication

import os
from dotenv import load_dotenv
from huggingface_hub import login

# Load environment variables from the .env file
load_dotenv()

# Read the Hugging Face token from the environment variable
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

# Log in using the token
login(token=huggingface_token)

Hugging Face token loaded successfully.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /Users/sandordaroczi/.cache/huggingface/token
Login successful


In [1]:
# Checking CUDA availability

import torch

# Check CUDA availability
if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB


## 3. Loading Models

In [19]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# device = "cpu"
device = "cuda"

# model_name = "EleutherAI/gpt-neo-125m"  # Lightweight model for debugging purposes
# model_name = "meta-llama/Meta-Llama-3-8B"  # Too large to run on a gpu_gtx1080. GPU gpu_a100 is required.
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small enough to run on a gpu_gtx1080.

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map=device)
model.NAME = model_name

print(f"Model Memory Footprint: {(model.get_memory_footprint() / (1024 ** 3)):.2f} GB")
# from src.models.utils_llm import calculate_model_size
# calculate_model_size(model_name)  # does not work, model saved in a different location
if torch.cuda.is_available():
  from src.models.utils_llm import print_gpu_utilization
  print_gpu_utilization()

Model Memory Footprint: 4.10 GB
GPU memory occupied: 13.47 GB.


In [19]:
# Example inference

input_text = "What famous tower is in Paris?"
input_ids = tokenizer(input_text, return_tensors="pt").to(device)

generated_ids = model.generate(
    input_ids=input_ids["input_ids"],
    max_length=512,  # Adjust maximum output length
    num_beams=5,  # Adjust number of beams for beam search
)

generated_text = tokenizer.decode(generated_ids[0][:1000], skip_special_tokens=True)
print("Generated Text:", generated_text)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Generated Text: What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in Paris?

What famous tower is in 

In [20]:
generated_ids[0][:1000]

tensor([ 2061,  5863, 10580,   318,   287,  6342,    30,   198,   198,  2061,
         5863, 10580,   318,   287,  6342,    30,   198,   198,  2061,  5863,
        10580,   318,   287,  6342,    30,   198,   198,  2061,  5863, 10580,
          318,   287,  6342,    30,   198,   198,  2061,  5863, 10580,   318,
          287,  6342,    30,   198,   198,  2061,  5863, 10580,   318,   287,
         6342,    30,   198,   198,  2061,  5863, 10580,   318,   287,  6342,
           30,   198,   198,  2061,  5863, 10580,   318,   287,  6342,    30,
          198,   198,  2061,  5863, 10580,   318,   287,  6342,    30,   198,
          198,  2061,  5863, 10580,   318,   287,  6342,    30,   198,   198,
         2061,  5863, 10580,   318,   287,  6342,    30,   198,   198,  2061,
         5863, 10580,   318,   287,  6342,    30,   198,   198,  2061,  5863,
        10580,   318,   287,  6342,    30,   198,   198,  2061,  5863, 10580,
          318,   287,  6342,    30,   198,   198,  2061,  5863, 

In [ ]:
import json

results = {"input": input_text, "output": generated_text}
with open("results.json", "w") as f:
    json.dump(results, f)

## 4. Loading Datasets

### 4.1. WikiText

In [5]:
# Initialize the datamodule
import os
from src.data.WikiTextDataModule import WikiTextDataModule

directory_dataset = os.getcwd()
wikitext_batch_size = 1  # Just use batch size 1 for this project
#wikitext_batch_size = 16
#wikitext_batch_size = 64
sequence_length = 512  # Maximum sequence length - use the default value
wikitext_seed = 1

wikitext_data_module = WikiTextDataModule(
  directory_dataset=directory_dataset,
  batch_size=wikitext_batch_size,
  sequence_length=sequence_length,
  tokenizer_name=model_name,
  seed=wikitext_seed
)

#wikitext_train_dataloader = wikitext_data_module.train_dataloader()
wikitext_dataloader = wikitext_data_module.val_dataloader()

Token indices sequence length is longer than the specified maximum sequence length for this model (294896 > 2048). Running this sequence through the model will result in indexing errors


In [8]:
print("Length of datasets:", len(wikitext_data_module.train_dataset), len(wikitext_data_module.val_dataset), len(wikitext_data_module.test_dataset))

# Reason why the numbers are low: number of tokens / 2048 -> gives the number of elements in the dataset

dataset_size = len(wikitext_dataloader)
print(f"Number of batches in train_dataloader: {dataset_size}")
for i, (data, target) in enumerate(wikitext_dataloader):
    if i < 2:
        print(f"Batch {i+1}:")
        original_text = tokenizer.decode(data[0], skip_special_tokens=True)
        print(f"  Original Text: {original_text[:500]}")
        print(f"  Input data (first 5 tokens): {data[0][:5]}")
        print(f"  Target labels (first 5 tokens): {target[0][:5]}")
        print(f"  Input data shape: {data.shape}")
        print(f"  Target labels shape: {target.shape}")


Length of datasets: 36718 3760 4358
Number of batches in train_dataloader: 498
Batch 1:
  Original Text:   = Homarus gammarus = 
   Homarus gammarus, known as the European lobster or common lobster, is a species of clawed lobster from the eastern Atlantic Ocean, Mediterranean Sea and parts of the Black Sea. It is closely related to the American lobster, H. americanus. It may grow to a length of 60 cm ( 24 in ) and a mass of 6 kilograms ( 13 lb ), and bears a conspicuous pair of claws. In life, the lobsters are blue, only becoming " lobster red " on cooking. Mating occurs in the summer, producing eg
  Input data (first 5 tokens): tensor([128000,    220,    284,  13525,  51419])
  Target labels (first 5 tokens): tensor([  220,   284, 13525, 51419,  9192])
  Input data shape: torch.Size([1, 512])
  Target labels shape: torch.Size([1, 512])
Batch 2:
  Original Text:  the American lobster, Homarus americanus. The two species are very similar, and can be crossed artificially, although hybrid

In [6]:
wikitext_dataset = []

# Loop through each batch in the dataloader
for batch in wikitext_dataloader:
  # Assuming the batch contains input_ids (tokenized text) and labels
  input_ids, labels = batch
  
  # Decode the input IDs back to text using the tokenizer
  decoded_text = wikitext_data_module.tokenizer.decode(input_ids[0].tolist())  # Assuming first element in batch
  wikitext_dataset.append(decoded_text)
  
print(f"Sample texts from train dataloader: {wikitext_dataset[1][:1000]}")
print(f"Type: {type(wikitext_dataset)}, Length: {len(wikitext_dataset)}")


Sample texts from train dataloader: , with spots that coalesce , and yellow below . The red colour associated with lobsters only appears after cooking . This occurs because , in life , the red pigment astaxanthin is bound to a protein complex , but the complex is broken up by the heat of cooking , releasing the red pigment . 
  The closest relative of H. gammarus is the American lobster , Homarus americanus . The two species are very similar , and can be crossed artificially , although hybrids are unlikely to occur in the wild since their ranges do not overlap . The two species can be distinguished by a number of characteristics : 
  The rostrum of H. americanus bears one or more spines on the underside , which are lacking in H. gammarus . 
  The spines on the claws of H. americanus are red or red @-@ tipped , while those of H. gammarus are white or white @-@ tipped . 
  The underside of the claw of H. americanus is orange or red , while that of H. gammarus is creamy white or very pale

### 4.2. OpenAssistant

In [22]:
# Initialize the datamodule
import os
from src.data.OpenAssistantDataModule import OpenAssistantDataModule

directory_dataset = os.getcwd()
oasst_batch_size = 1  # Just use batch size 1 for this project
# oasst_batch_size = 16
# oasst_batch_size = 64
oasst_sequence_length = 512  # Maximum sequence length - use the default value
oasst_seed = 1

# Data Module
oasst_data_module = OpenAssistantDataModule(
  directory_dataset=directory_dataset,
  batch_size=oasst_batch_size,
  sequence_length=oasst_sequence_length,
  tokenizer_name=model_name,
  seed=oasst_seed
)

# Data Loader
# oasst_dataloader = oasst_data_module.train_dataloader()
oasst_dataloader = oasst_data_module.val_dataloader()

Token indices sequence length is longer than the specified maximum sequence length for this model (647957 > 2048). Running this sequence through the model will result in indexing errors


In [17]:
print(f"Length of datasets:", len(oasst_data_module.train_dataset), len(oasst_data_module.val_dataset))

# Reason why the numbers are low: number of tokens / 2048 -> gives the number of elements in the dataset

oasst_dataset_size = len(oasst_dataloader)
print(f"Number of batches in train_dataloader: {dataset_size}")
for i, (data, target) in enumerate(oasst_dataloader):
    if i < 2:
        print(f"Batch {i+1}:")
        original_text = tokenizer.decode(data[0], skip_special_tokens=True)
        print(f"  Original Text: {original_text[:500]}")
        print(f"  Input data (first 5 tokens): {data[0][:5]}")
        print(f"  Target labels (first 5 tokens): {target[0][:5]}")
        print(f"  Input data shape: {data.shape}")
        print(f"  Target labels shape: {target.shape}")

Length of datasets: 84437 4401
Number of batches in train_dataloader: 498
Batch 1:
  Original Text: Напиши функцию на языке swift, которая сортирует массив целых чисел, а затем выводит его на экран Вот функция, которая сортирует массив целых чисел и выводит его на экран:

```swift
func sortAndPrintArray(_ array: [Int]) {
  // Создаем копию массива, чтобы не изменять исходный
  var sortedArray = array
  // Сортируем массив по возрастанию
  sortedArray.sort()
  // Выводим отсортированный массив на экран
  print(sortedArray)
}
```


Ты можешь проверить работу функции, вызвав ее с любым массивом ц
  Input data (first 5 tokens): tensor([128000,  20807,  19619,   1840,  30480])
  Target labels (first 5 tokens): tensor([20807, 19619,  1840, 30480, 55813])
  Input data shape: torch.Size([1, 512])
  Target labels shape: torch.Size([1, 512])
Batch 2:
  Original Text:  boca está llena de afilados dientes y mandíbulas capaces de morder y desgarrar cualquier cosa que se interponga en su camino.

Ve

In [8]:
oasst_dataset = []

# Loop through each batch in the dataloader
for batch in oasst_dataloader:
  # Assuming the batch contains input_ids (tokenized text) and labels
  input_ids, labels = batch
  
  # Decode the input IDs back to text using the tokenizer
  decoded_text = oasst_data_module.tokenizer.decode(input_ids[0].tolist())  # Assuming first element in batch
  oasst_dataset.append(decoded_text)
  
print(f"Sample texts from train dataloader: {oasst_dataset[1][:1000]}")
print(f"Type: {type(oasst_dataset)}, Length: {len(oasst_dataset)}")


Sample texts from train dataloader:  boca está llena de afilados dientes y mandíbulas capaces de morder y desgarrar cualquier cosa que se interponga en su camino.

Ventajas:

• Xalakthrax es un monstruo altamente adaptable a diferentes entornos, gracias a su exoesqueleto segmentado que le permite modificar su forma y tamaño según las necesidades.

• Sus espinas venenosas pueden incapacitar o incluso matar a sus presas, lo que le convierte en un depredador letal.

• Su mandíbula es extremadamente fuerte y sus dientes afilados le permiten alimentarse de cualquier tipo de carne.

Desventajas:

• El cuerpo altamente complejo de Xalakthrax lo hace vulnerable a ataques en áreas no protegidas por su exoesqueleto, como su cuello o su abdomen.

• Xalakthrax necesita alimentarse con frecuencia debido a su alto metabolismo y a su gran tamaño, lo que lo hace vulnerable a periodos de escasez de comida.

• Xalakthrax es un monstruo solitario que tiene dificultades para relacionarse con otros seres, 

## 5. Quantization

### 5.1. BitsAndBytes

#### 5.1.1 BitsAndBytes 8-bit

In [21]:
# BNB Config 8-bit

from transformers import BitsAndBytesConfig

from src import MODEL_SAVE_PATH

# Define the device and model name
device = "cuda"
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# BnB Quantization Configurations
bnb_config_8bit = BitsAndBytesConfig(
    load_in_8bit=True,
    load_in_4bit=False,
    llm_int8_threshold=6.0,
    llm_int8_enable_fp32_cpu_offload=False,
    llm_int8_has_fp16_weight=False,
)

# Save path
import os
bnb_8bit_model_name = f"{model_name.split('/')[1]}-bnb-8bit"
bnb_8bit_model_path = os.path.join(MODEL_SAVE_PATH, bnb_8bit_model_name)
os.makedirs(bnb_8bit_model_path, exist_ok=True)

In [22]:
# Quantization 8-bit

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
model_bnb_8bit = AutoModelForCausalLM.from_pretrained(
    model_name, 
    quantization_config=bnb_config_8bit, 
    torch_dtype=torch.float32,
    device_map=device
)
model_bnb_8bit.NAME = bnb_8bit_model_name

print(f"8-bit BNB Model Memory Footprint: {(model_bnb_8bit.get_memory_footprint() / (1024 ** 3)):.2f} GB")

from accelerate import Accelerator
accelerate = Accelerator()
accelerate.save_model(model_bnb_8bit, bnb_8bit_model_path)

print(f"8-bit BnB model saved at: {bnb_8bit_model_path}")

from src.models.utils_llm import calculate_model_size
calculate_model_size(bnb_8bit_model_path)
from src.models.utils_llm import print_gpu_utilization
print_gpu_utilization()

OutOfMemoryError: CUDA out of memory. Tried to allocate 250.00 MiB. GPU 

#### 5.1.2 BitsAndBytes 4-bit

In [ ]:
# BNB Config 4-bit

import torch
from transformers import BitsAndBytesConfig

from src import MODEL_SAVE_PATH

# Define the device and model name
device = "cuda"
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config_4bit = BitsAndBytesConfig(
    load_in_8bit=False,
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="fp4",
    bnb_4bit_use_double_quant=False,
)

# Save path
import os
bnb_4bit_model_name = f"{model_name.split('/')[1]}-bnb-4bit"
bnb_4bit_model_path = os.path.join(MODEL_SAVE_PATH, bnb_4bit_model_name)

In [ ]:
# Quantization 4-bit

from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
model_bnb_4bit = AutoModelForCausalLM.from_pretrained(
    model_name, 
    quantization_config=bnb_config_4bit, 
    torch_dtype=torch.float32,
    device_map=device
)
model_bnb_4bit.NAME = bnb_4bit_model_name

print(f"4-bit BNB Model Memory Footprint: {(model_bnb_4bit.get_memory_footprint() / (1024 ** 3)):.2f} GB")

from accelerate import Accelerator
accelerate = Accelerator()
accelerate.save_model(model_bnb_4bit, bnb_4bit_model_path)

print(f"4-bit BnB model saved at: {bnb_4bit_model_path}")

from src.models.utils_llm import calculate_model_size
calculate_model_size(bnb_4bit_model_path)
from src.models.utils_llm import print_gpu_utilization
print_gpu_utilization()

4-bit BNB Model Memory Footprint: 0.94 GB
4-bit BnB model saved at: /nfs/students/daro/models/TinyLlama-1.1B-Chat-v1.0-bnb-4bit
Total Model Size for /nfs/students/daro/models/TinyLlama-1.1B-Chat-v1.0-bnb-4bit: 1020.20 MB
GPU memory occupied: 37.64 GB.


### 5.2 AWQ

In [ ]:
# AWQ Config
awq_config = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM"
}

# AWQ Calibration Split
awq_calib_split = "validation"

# Save path
import os
awq_model_name = f"{model_name.split('/')[1]}-awq"
awq_model_path = os.path.join(MODEL_SAVE_PATH, awq_model_name)

# Define the device and model name
device = "cuda"
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [4]:
# AWQ Quantization

from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

awq_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
awq_model = AutoAWQForCausalLM.from_pretrained(
    model_name,
    device_map=device
)

# Quantize with wikitext validation as calibration data
awq_model.quantize(
    tokenizer=tokenizer,
    quant_config=awq_config,
    calib_data=wikitext_dataset,  # Pass the loaded validation dataset here
)
awq_model.NAME = awq_model_name

# Save quantized model
awq_model.save_quantized(awq_model_path)
awq_tokenizer.save_pretrained(awq_model_path)

print(f'Model is quantized and saved at "{awq_model_path}"')

from src.models.utils_llm import calculate_model_size
calculate_model_size(awq_model_path)
from src.models.utils_llm import print_gpu_utilization
print_gpu_utilization()

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 1002.00 MiB. GPU 

In [ ]:
# Load model and generate text
awq_model_path = "TinyLlama-1.1B-Chat-v1.0-awq"

awq_tokenizer = AutoTokenizer.from_pretrained(awq_model_path)
awq_model = AutoAWQForCausalLM.from_pretrained(
    awq_model_path,
    trust_remote_code=True,
)

# Generate text
prompt = "What is the weather like today?"
input_ids = awq_tokenizer.encode(prompt, return_tensors="pt")  # Convert text to tensors

output = awq_model.generate(input_ids)
generated_text = awq_tokenizer.decode(output[0], skip_special_tokens=True)

### 5.3 HQQ

In [ ]:
# HQQ Config

from transformers import AutoTokenizer

# Define the model name and tokenizer
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)

### 5.3.1 Option 1: All linear layers will use the same quantization config

In [ ]:
import torch
from transformers import AutoModelForCausalLM, HqqConfig

# Option 1: All linear layers will use the same quantization config
quant_config_same = HqqConfig(
    nbits=8, 
    group_size=64, 
    quant_zero=False, 
    quant_scale=False, 
    axis=0  # Default value
)

# Quantize the model with the same config for all linear layers
model_same = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda",
    quantization_config=quant_config_same
)

# Save the quantized model with the same config for all layers
model_same_path = f"{model_name}-hqq-same"
model_same.save_pretrained(model_same_path)
tokenizer.save_pretrained(model_same_path)
print(f"Model with the same quantization config saved at '{model_same_path}'")

from src.models.utils_llm import calculate_model_size
print(f"Model size (same config): {calculate_model_size(model_same_path)}")

### 5.3.2. Option 2: Different configs for specific layers

In [ ]:
import torch
from transformers import AutoModelForCausalLM, HqqConfig

# Option 2: Different configs for specific layers
q4_config = {'nbits': 4, 'group_size': 64, 'quant_zero': False, 'quant_scale': False}
q3_config = {'nbits': 3, 'group_size': 32, 'quant_zero': False, 'quant_scale': False}

quant_config_dynamic = HqqConfig(dynamic_config={
    'self_attn.q_proj': q4_config,
    'self_attn.k_proj': q4_config,
    'self_attn.v_proj': q4_config,
    'self_attn.o_proj': q4_config,
    'mlp.gate_proj': q3_config,
    'mlp.up_proj': q3_config,
    'mlp.down_proj': q3_config,
})

# Quantize the model with different configs for specific layers
model_dynamic = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda",
    quantization_config=quant_config_dynamic
)

# Save the quantized model with different configs for specific layers
model_dynamic_path = f"{model_name}-hqq-dynamic"
model_dynamic.save_pretrained(model_dynamic_path)
tokenizer.save_pretrained(model_dynamic_path)
print(f"Model with dynamic quantization config saved at '{model_dynamic_path}'")

from src.models.utils_llm import calculate_model_size
print(f"Model size (dynamic config): {calculate_model_size(model_dynamic_path)}")

## 6. Evaluation

### 6.1. Perplexity

In [ ]:
import torch
import tqdm

def evaluate_perplexity(model, tokenizer, data_module, data_split="validation", max_length=None, stride=512, factor=1, to_device=False, device="cuda"):
    model.eval()
    dataset_map = {
        "train": data_module.train_dataset,
        "validation": data_module.val_dataset,
        "test": data_module.test_dataset
    }
    
    if max_length is None:
        max_length = tokenizer.model_max_length
    if to_device:
        model.to(device)
        
    encodings = tokenizer("\n\n".join(dataset_map[data_split]["text"]), return_tensors="pt")
    seq_len = encodings.input_ids.size(1)

    nlls = []
    prev_end_loc = 0
    for begin_loc in tqdm.tqdm(range(0, seq_len//factor, stride)):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc  # may be different from stride on last loop
        input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():
            outputs = model(input_ids, labels=target_ids)

            # loss is calculated using CrossEntropyLoss which averages over valid labels
            # N.B. the model only calculates loss over trg_len - 1 labels, because it internally shifts the labels
            # to the left by 1.
            neg_log_likelihood = outputs.loss

        nlls.append(neg_log_likelihood)
        prev_end_loc = end_loc
        if end_loc == seq_len:
            break

    ppl = torch.exp(torch.stack(nlls).mean())
    print(f"Perplexity of model {model.NAME}: {ppl:.2f}")
    
    return ppl

In [59]:
evaluate_perplexity(model, tokenizer, wikitext_data_module, device="cuda")

Token indices sequence length is longer than the specified maximum sequence length for this model (299964 > 2048). Running this sequence through the model will result in indexing errors
100%|██████████| 6/6 [00:01<00:00,  3.87it/s]


Perplexity of model TinyLlama/TinyLlama-1.1B-Chat-v1.0: 5.76


tensor(5.7600, device='cuda:0')

In [60]:
evaluate_perplexity(model_bnb_8bit, tokenizer, wikitext_data_module, device=device)

  0%|          | 0/6 [00:00<?, ?it/s]/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:316: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
100%|██████████| 6/6 [00:01<00:00,  4.12it/s]

Perplexity of model TinyLlama-1.1B-Chat-v1.0-bnb-8bit: 5.77


tensor(5.7749, device='cuda:0')

In [61]:
evaluate_perplexity(model_bnb_4bit, tokenizer, wikitext_data_module, device=device)

100%|██████████| 6/6 [00:01<00:00,  3.74it/s]


Perplexity of model TinyLlama-1.1B-Chat-v1.0-bnb-4bit: 6.19


tensor(6.1906, device='cuda:0')

In [65]:
awq_model.NAME = awq_model_name
evaluate_perplexity(awq_model, tokenizer, wikitext_data_module, to_device=True, device=device)

100%|██████████| 6/6 [00:00<00:00,  8.46it/s]

Perplexity of model TinyLlama-1.1B-Chat-v1.0-awq: 6.00


tensor(6.0030, device='cuda:0')

In [63]:
print(len(tokenizer.vocab))
print(len(awq_tokenizer.vocab))

32000
32000


In [ ]:
evaluate_perplexity(awq_model, wikitext_dataloader, device="cuda")

In [ ]:
# Evaluate perplexity on each model
from src.evaluations.evaluate_text_generation import evaluate_perplexity

list_of_models = [model, model_bnb_8bit, model_bnb_4bit]
results = {}
for model in list_of_models:
  print(f"Perplexity for model {model.NAME}: {evaluate_perplexity(model_bnb_8bit, wikitext_data_module, device)}"

# Print perplexity results
#print(f"Perplexity (8-bit): {perplexity_8bit:.4f}")
#print(f"Perplexity (4-bit): {perplexity_4bit:.4f}")
print(f"Perplexity (Original): {perplexity_original:.4f}")

### 6.2. Brier Score

In [7]:
import torch
import tqdm
import torch.nn.functional as F

def evaluate_brier_score(model, tokenizer, dataloader, max_length=None, stride=512, factor=1, to_device=False, device="cuda"):
    model.eval()
    
    if max_length is None:
        max_length = tokenizer.model_max_length
    if to_device:
        model.to(device)
        
    encodings = tokenizer("\n\n".join(dataloader.dataset.dataset["text"]), return_tensors="pt")
    seq_len = encodings.input_ids.size(1)

    brier_scores = []
    prev_end_loc = 0
    for begin_loc in tqdm.tqdm(range(0, seq_len//factor, stride)):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc  # may be different from stride on last loop
        input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
        target_ids = input_ids.clone().to(device)
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():
            outputs = model(input_ids)
            logits = outputs.logits
            
            # Shift logits and target_ids to the left by 1 for calculating the Brier score
            shifted_logits = logits[:, :-1].contiguous()
            shifted_target_ids = target_ids[:, 1:].contiguous()

            # Flatten the logits and target_ids for calculation
            shifted_logits = shifted_logits.view(-1, shifted_logits.size(-1))
            shifted_target_ids = shifted_target_ids.view(-1)

            # Filter out the -100 targets
            valid_indices = shifted_target_ids != -100
            valid_logits = shifted_logits[valid_indices]
            valid_target_ids = shifted_target_ids[valid_indices]

            # Get the probabilities
            probs = F.softmax(valid_logits, dim=-1)

            # Create one-hot target vectors
            targets = F.one_hot(valid_target_ids, num_classes=probs.size(-1)).float()

            # Calculate the Brier score
            brier_score = torch.mean((probs - targets) ** 2)
            brier_scores.append(brier_score)

    avg_brier_score = torch.stack(brier_scores).mean()
    print(f"Brier Score of model {model.NAME}: {avg_brier_score:.4f}")
    
    return avg_brier_score

In [8]:
evaluate_brier_score(model, tokenizer, wikitext_dataloader, factor=100, device=device)

Token indices sequence length is longer than the specified maximum sequence length for this model (299964 > 2048). Running this sequence through the model will result in indexing errors
100%|██████████| 6/6 [00:02<00:00,  2.48it/s]

Brier Score of model TinyLlama/TinyLlama-1.1B-Chat-v1.0: 0.0000


tensor(1.6116e-05, device='cuda:0')

In [14]:
evaluate_brier_score(model_bnb_4bit, tokenizer, wikitext_data_module, device=device)

Token indices sequence length is longer than the specified maximum sequence length for this model (299964 > 2048). Running this sequence through the model will result in indexing errors
100%|██████████| 6/6 [00:01<00:00,  3.10it/s]

Brier Score of model TinyLlama-1.1B-Chat-v1.0-bnb-4bit: 0.0000


tensor(1.6541e-05, device='cuda:0')

In [15]:
evaluate_brier_score(model_bnb_8bit, tokenizer, wikitext_data_module, device=device)

  0%|          | 0/6 [00:00<?, ?it/s]/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:316: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
100%|██████████| 6/6 [00:01<00:00,  3.11it/s]

Brier Score of model TinyLlama-1.1B-Chat-v1.0-bnb-8bit: 0.0000


tensor(1.6136e-05, device='cuda:0')

In [16]:
evaluate_brier_score(awq_model, tokenizer, wikitext_data_module, device=device)

NameError: name 'awq_model' is not defined

# 7. SEML Pipeline

In [9]:
import shutil
import re
import os
import torch

# To avoid the following problem when running seml (see https://github.com/pytorch/pytorch/issues/37377)
os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
if re.match(".*username.*", os.getcwd()):
    CACHE_PATH = "~/.cache/"
else:
    CACHE_PATH = "/tmp/"

torch.hub.set_dir(CACHE_PATH)

import logging
logger = logging.getLogger("quant_logger")

#os.chdir('..')
print("Current Working Directory " , os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name
import seml

# Reload the src module after making changes
importlib.reload(src)

from src.models import get_model, get_model_name
from src.data import get_dataset, get_data_loader_from_split
from src.algorithms.quantization.quantize import quantize
from src.evaluations.evaluate_all import evaluate

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability


In [2]:
evaluate.__code__.co_varnames

('model',
 'eval_tokenizer',
 'eval_data_module',
 'eval_metrics',
 'stride',
 'factor',
 'device',
 'prefix',
 'results')

In [12]:
def run_quantize(
    # Dataset parameters
    seed_dataset=123,
    directory_dataset="",
    calib_dataset_name="",
    calib_dataset_split="",
    eval_dataset_name="",
    eval_dataset_split="",
    batch_size=1,
    stride=512,
    # Model parameters
    seed_model=123,
    directory_model="",
    clean_cache=True,
    model_name="",
    # Quantization parameters
    quantize_method="",
    quantize_params={},
    # Evaluation metrics
    eval_metrics=[
        "perplexity",
    ],
    device="cuda",
    save_quantized_model=False,
    quantized_model_save_path="",
):
    ##################
    ## Print config ##
    ##################
    logger.info("Received the following configuration:")
    logger.info(
        f"Calibration dataset: {calib_dataset_name}\n"
        f"Calibration split: {calib_dataset_split}\n"
        f"Evaluation dataset: {eval_dataset_name}\n"
        f"Evaluation split: {eval_dataset_split}\n"
        f"Batch size: {batch_size}\n"
        f"Model: {model_name}\n"
        f"Quantize method: {quantize_method}\n"
        f"Quantize params: {quantize_params}\n"
        f"Evaluation metrics: {eval_metrics}\n"
        f"Device: {device}\n"
    )
    
    ################
    ## Load model ##
    ################
    logger.info("Load base model")
    model_full_name = get_model_name(model_name)
    model, tokenizer = get_model(
        model_name=model_full_name,
        seed=seed_model,
        directory_model=directory_model,
        device=device,
    )
    
    ###############
    ## Load data ##
    ###############
    logger.info("Load calibration and evaluation data modules")
    calib_data_module = get_dataset(
        dataset_name=calib_dataset_name,
        directory_dataset=directory_dataset,
        batch_size=batch_size,
        sequence_length=stride,
        tokenizer_name=model_full_name,
        seed=seed_dataset,
    )
    eval_data_module = get_dataset(
        dataset_name=eval_dataset_name,
        directory_dataset=directory_dataset,
        batch_size=batch_size,
        sequence_length=stride,
        tokenizer_name=model_full_name,
        seed=seed_dataset,
    )
    
    calib_tokenizer = calib_data_module.tokenizer
    calib_dataloader = get_data_loader_from_split(calib_data_module, calib_dataset_split)
    
    eval_tokenizer = eval_data_module.tokenizer
    eval_dataloader = get_data_loader_from_split(eval_data_module, eval_dataset_split)

    ################################
    ## Update quantize parameters ##
    ################################
    logger.info("Defining quantize parameters")
    quantize_params.update(quantize_params)
    logger.info(f"Default parameters adjusted from {quantize_params}")

    ##############
    ## Quantize ##
    ##############
    logger.info("Quantization")
    quantized_model, quantized_tokenizer = quantize(
        model_name=model_full_name,
        calib_tokenizer=calib_tokenizer,
        calib_dataloader=calib_dataloader,
        quantize_method=quantize_method,
        quantize_config=quantize_params,
        save_model=save_quantized_model,
        save_path=quantized_model_save_path,
        device=device
    )

    ##############
    ## Evaluate ##
    ##############
    logger.info("Evaluating the quantized models")
    results = evaluate(
        model=quantized_model,
        eval_tokenizer=eval_tokenizer,
        eval_dataloader=eval_dataloader,
        eval_metrics=eval_metrics,
        stride=stride,
        factor=100,
        device=device,
        to_device=(quantize_method in ["AWQ"]),
        prefix="",
    )

    ####################
    ## Cleaning cache ##
    ####################
    logger.info
    if clean_cache:
        for root, dirs, files in os.walk(CACHE_PATH, topdown=False):
            for dir_name in dirs:
                pattern = re.compile(f"^.*{model_full_name.split('/')[-1]}.*")
                dir_path = os.path.join(root, dir_name)
                if re.match(pattern, dir_path):
                    try:
                        shutil.rmtree(dir_path)
                    except:
                        pass

    fail_trace = {
        "fail_trace": seml.evaluation.get_results,
    }

    return {**results, **fail_trace}


In [13]:
import itertools
import random

# Fixed parameters
fixed_params = {
    'device': 'cuda',
    'clean_cache': True,
    'save_quantized_model': True,
    'seed_model': 123,
    'seed_dataset': 123,
    'batch_size': 1,
    'eval_metrics': ['perplexity', 'brier_score'],
    'calib_dataset_split': 'validation',
    'eval_dataset_split': 'test',
}

# Grid parameters
grid_params = {
    'calib_dataset_name': ['WikiText', 'OpenAssistant'],
    'eval_dataset_name': ['WikiText', 'OpenAssistant'],
    'quantize_method': ['BNB', 'AWQ'],
    'quantize_params': [
        {
            "num_bits": 8,
            "llm_int8_threshold": 6.0,
            "llm_int8_enable_fp32_cpu_offload": False,
            "llm_int8_has_fp16_weight": False
        },
        {
            "num_bits": 4,
            "bnb_4bit_compute_dtype": torch.bfloat16,
            "bnb_4bit_quant_type": "fp4",
            "bnb_4bit_use_double_quant": False,
        }
    ],
    'model_name': ['TinyLlama']
}

batch_sizes = [1]

# # Random parameters
# random_params = {
#     'samples': 2,
#     'seed': 12345,
#     'batch_size': {
#         'type': 'uniform',
#         'min': 1,
#         'max': 4
#     }
# }

# # Set seed for reproducibility
# random.seed(random_params['seed'])

# # Generate random batch sizes
# batch_sizes = [random.randint(random_params['batch_size']['min'], random_params['batch_size']['max']) for _ in range(random_params['samples'])]

# Generate all combinations for grid search
grid_combinations = list(itertools.product(
    grid_params['calib_dataset_name'],
    grid_params['eval_dataset_name'],
    grid_params['quantize_method'],
    grid_params['quantize_params'],
    grid_params['model_name']
))

# Run the quantize function for all combinations
results = []
max_combinations = 10000  # Set to a lower number for testing purposes
for i, combination in enumerate(grid_combinations):
    if i >= max_combinations:
        break
    for batch_size in batch_sizes:
        calib_dataset_name, eval_dataset_name, quantize_method, quantize_params, model_name = combination
        result = run_quantize(
            # Fixed parameters
            device=fixed_params['device'],
            clean_cache=fixed_params['clean_cache'],
            save_quantized_model=fixed_params['save_quantized_model'],
            seed_model=fixed_params['seed_model'],
            seed_dataset=fixed_params['seed_dataset'],
            eval_metrics=fixed_params['eval_metrics'],
            calib_dataset_split=fixed_params['calib_dataset_split'],
            eval_dataset_split=fixed_params['eval_dataset_split'],
            # Grid parameters
            calib_dataset_name=calib_dataset_name,
            eval_dataset_name=eval_dataset_name,
            quantize_method=quantize_method,
            quantize_params=quantize_params,
            model_name=model_name,
            # Random parameters
            batch_size=batch_size,
            stride=512,
            # Model parameters
            directory_model="",
            directory_dataset="",
            quantized_model_save_path=""
        )
        results.append(result)

# Do something with the results
print(results)


Token indices sequence length is longer than the specified maximum sequence length for this model (2824491 > 2048). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (2824491 > 2048). Running this sequence through the model will result in indexing errors


Model is quantized and saved at "/nfs/students/daro/models/TinyLlama-1.1B-Chat-v1.0-bnb-8bit"
Total Model Size for /nfs/students/daro/models/TinyLlama-1.1B-Chat-v1.0-bnb-8bit: 1.39 GB
GPU memory occupied: 10.78 GB.


  0%|          | 0/7 [00:00<?, ?it/s]/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:316: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
100%|██████████| 7/7 [00:01<00:00,  4.88it/s]


Perplexity of model TinyLlama-1.1B-Chat-v1.0-bnb-8bit: 8.85


100%|██████████| 7/7 [00:01<00:00,  5.08it/s]


...

Perplexity of model TinyLlama-1.1B-Chat-v1.0-awq: 9.17


100%|██████████| 7/7 [00:00<00:00,  8.88it/s]


Brier Score of model TinyLlama-1.1B-Chat-v1.0-awq: 0.0000


Token indices sequence length is longer than the specified maximum sequence length for this model (2824491 > 2048). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (12465711 > 2048). Running this sequence through the model will result in indexing errors


MisconfigurationException: `test_dataloader` must be implemented to be used with the Lightning Trainer

In [19]:
print(wikitext_data_module.val_dataset["text"][:100])
print(wikitext_dataloader.dataset.dataset["text"][:100])

print(len(wikitext_data_module.val_dataset["text"]))
print(len(wikitext_dataloader.dataset.dataset["text"]))

['', ' = Homarus gammarus = \n', '', ' Homarus gammarus , known as the European lobster or common lobster , is a species of clawed lobster from the eastern Atlantic Ocean , Mediterranean Sea and parts of the Black Sea . It is closely related to the American lobster , H. americanus . It may grow to a length of 60 cm ( 24 in ) and a mass of 6 kilograms ( 13 lb ) , and bears a conspicuous pair of claws . In life , the lobsters are blue , only becoming " lobster red " on cooking . Mating occurs in the summer , producing eggs which are carried by the females for up to a year before hatching into planktonic larvae . Homarus gammarus is a highly esteemed food , and is widely caught using lobster pots , mostly around the British Isles . \n', '', ' = = Description = = \n', '', ' Homarus gammarus is a large crustacean , with a body length up to 60 centimetres ( 24 in ) and weighing up to 5 – 6 kilograms ( 11 – 13 lb ) , although the lobsters caught in lobster pots are usually 23 – 38 cm ( 9 – 15